In [1]:
# import
import torch
import random
import torch.nn as nn
import torch.nn.functional as F

import warnings
warnings.filterwarnings("ignore")

seed = 333

# Hyperparameter

In [2]:
hp = {
    "t":8,
    "c":32,
    "hs":32,
}

# Data

## Import Data

In [3]:
# import data
with open("../data/input.txt","r",newline="\n") as file :
    data_file = file.read()

data_file[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

## Data Class

In [4]:
class Data:
    def __init__(
        self,
        _data: str,
        seed: int = seed,
        train_batch_size: int = None,
        train_block_size: int = None,
        test_batch_size: int = None,
        test_block_size: int = None,
    ) -> None:
        # initialisation
        self.seed = seed
        self.data = _data  # raw data without process
        self.len_data = len(_data)  # how many character in raw data
        self.char_list = sorted(list(set(_data)))  # list of possible character
        self.len_char = len(self.char_list)  # len of possible character

        # encoder and decoder list
        self.enc_dict = {key: value for value, key in enumerate(self.char_list)}  # convert to number
        self.dec_dict = {key: value for key, value in enumerate(self.char_list)}  # convert to string

        # encoded data
        self.data_enc = self.encoder(self.data)  # raw data without process

        # split to train and test set with ration 8:2
        split_at_index = self.len_data // 8
        self.train = self.data_enc[:split_at_index]  # full train set
        self.test = self.data_enc[split_at_index:]  # full test set

        # create x,y for train and test set
        if not None in [train_batch_size, test_batch_size, train_block_size, test_block_size]:
            self.x_train, self.y_train = self.sample_batch(
                batch_size=train_batch_size, block_size=train_block_size, input=self.train
            )
            self.x_test, self.y_test = self.sample_batch(
                batch_size=test_batch_size, block_size=test_block_size, input=self.test
            )

    def encoder(self, input: str) -> torch.Tensor:
        """convert string to encoded index

        Args:
            input (str): string to be encoded

        Returns:
            torch.Tensor: (copy) tensor of encoded index
        """
        return torch.tensor([self.enc_dict[char] for char in input])

    def decoder(self, input: torch.Tensor) -> str:
        """covnert encoded index in torch.tensor into

        Args:
            input (torch.Tensor): tensor of encoded index

        Returns:
            str: (copy) string from encoded index in tensor(input)
        """
        return "".join([self.dec_dict[i.item()] for i in input])

    def sample_batch(self, batch_size: int, block_size: int, input: torch.Tensor):
        """generate sample batch

        Args:
            batch_size (int): how many batch generated
            block_size (int): maximum length per batch
            input (torch.Tensor): tensor of encoded data

        Returns:
            x,y: data_x,data_y
        """
        output_x = torch.empty((batch_size, block_size), dtype=torch.long)
        output_y = torch.empty((batch_size, block_size), dtype=torch.long) if block_size !=1 else torch.empty((batch_size), dtype=torch.long)
        len_input = len(input)

        for batch_index in range(batch_size):
            offset = random.randint(0, len_input - block_size - 1)
            output_x[batch_index] = input[offset : offset + block_size]
            output_y[batch_index] = input[offset + 1 : offset + block_size + 1]
        return output_x, output_y

In [5]:
d = Data(
    _data=data_file,
    seed=seed,
    train_batch_size=50000,
    test_batch_size=50000,
    train_block_size=hp["t"],
    test_block_size=hp["t"],
)

# Layer/Block

In [34]:
class Head(nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        torch.manual_seed(seed)

        # layers
        self.query = nn.Linear(hp['c'],hp['hs'])
        self.key = nn.Linear(hp['c'],hp['hs'])
        self.value = nn.Linear(hp['c'],hp['hs'])

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        """forward prop

        Args:
            input (torch.Tensor): feed in embedded input data,shape(b,t,c)

        Returns:
            torch.Tensor: output tensor with shape # (b,t,hs)
        """
        # get q,k,v
        q = self.query(input)  # shape: (b,t, hs)
        k = self.key(input)  # shape: (b,t, hs)
        v = self.value(input)  # shape: (b,t, hs)
        
        # get attention
        attention_weight = (q @ k.transpose(-2,-1)) * (hp['hs'] ** -0.5)  # shape: (b,t,t)
        now_b,now_t,now_c = attention_weight.shape
        self.register_buffer("tril", torch.ones((now_t,now_c)).triu(diagonal=1).bool())
        attention_weight = attention_weight.masked_fill(self.tril, float('-inf'))  # shape: (b,t,t)
        attention_weight = F.softmax(attention_weight,dim=-1)  # shape: (b,t,t)
        
        # output
        out = attention_weight @ v  # shape: (b,t,hs)
        
        return out

# Main Model

In [36]:
class Model(nn.Module):
    def __init__(self):
        # initialisation
        super().__init__()
        torch.manual_seed(seed)

        # embedding
        self.embedding = nn.Embedding(d.len_char, hp["c"])
        self.head = Head()
        self.linear = nn.Linear(hp["hs"], d.len_char)
        self.flatten = nn.Flatten(0, 1)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        """forward prop

        Args:
            input (torch.Tensor): input tensor with shape (batch_size,block_size)

        Returns:
            torch.Tensor: logit tensor with shape (batch_size,len_char=65)
        """
        emb_out: torch.Tensor = self.embedding(input)  # shape = (b,t,c)
        head_out: torch.Tensor = self.head(emb_out)  # shape = (b,t,hs)
        logits: torch.Tensor = self.linear(head_out)  # shape = (b,t,len_char)
        if logits.ndim == 3:
            logits: torch.Tensor = self.flatten(logits)  # shape = (b*t,len_char)

        return logits

    def loss(self, logits: torch.Tensor, expected: torch.Tensor):
        """calculate loss

        Args:
            logits (torch.Tensor): logit tensor with size (batch_size,len_char)
            expected (torch.Tensor): expected class tensor with shape (batch size,categorical number within len_char)

        Returns:
            loss_value in tensor
        """
        return F.cross_entropy(logits, expected.view(-1))

    @torch.no_grad()
    def loss_test(self):
        """calculate loss of test set

        Returns:
            loss value of test set
        """
        return self.loss(self(d.x_test), d.y_test).item()

    def fit(
        self,
        x: torch.Tensor,
        y: torch.Tensor,
        lr: float = 1e-3,
        lr_step_size: int = 1000,
        lr_gamma: float = 0.9,
        batch_size: int = 100,
        iteration: int = 10000,
        print_iteration: int = 1000,
    ) -> None:
        # optimizer
        optimiser = torch.optim.AdamW(self.parameters(), lr=lr)
        lr_decay = torch.optim.lr_scheduler.StepLR(optimiser, step_size=lr_step_size, gamma=lr_gamma)
        
        for i in range(iteration):
            # batch x,y
            batch_index = torch.randint(low=0, high=y.shape[0], size=(batch_size,))
            batch_x = x[batch_index]
            batch_y = y[batch_index]
            batch_y = batch_y.view(-1)

            loss = self.loss(logits=self(batch_x), expected=batch_y)
            optimiser.zero_grad(set_to_none=True)
            loss.backward()
            optimiser.step()
            lr_decay.step()

            if ((i + 1) % print_iteration == 0) or ((i + 1) == iteration) or (i == 0):
                if i == 0:
                    print("                      Loss_Train : Loss test")
                with torch.no_grad():
                    print(f"{i+1}th Iteration = {self.loss(self.forward(x), y):.4f} : {self.loss_test():.4f}")

    @torch.no_grad()
    def sampling(self, idx, number_of_word: int):
        """generate sample

        Args:
            idx (torch.tensor): shape = (t)
            number_of_word (int): number of word to be generated
        """
        self.eval() # eva mode
        
        out = idx
        b, t = idx.shape

        for _ in range(number_of_word):
            # get last char
            last_char_enc = out[:, -hp["t"] :]

            # forward
            logits = self(last_char_enc)[-1]
            softmax = F.softmax(logits)

            # get next char
            next_char_enc = torch.multinomial(softmax, 1)

            # append
            out = torch.cat((out, next_char_enc.view(b, 1)), dim=-1)
        print(d.decoder(out.view(-1)))
        
        self.train() # train mode

In [32]:
m = Model()
m.fit(
    x=d.x_train,
    y=d.y_train,
    lr=1e-3,
    lr_step_size=2000,
    lr_gamma=1, # set to 1 to disable
    batch_size=32,
    iteration=20000,
    print_iteration=1000,
)


                      Loss_Train : Loss test
1th Iteration = 4.1577 : 4.1590
1000th Iteration = 2.4481 : 2.6537
2000th Iteration = 2.3872 : 2.6511
3000th Iteration = 2.3643 : 2.6731
4000th Iteration = 2.3486 : 2.6783
5000th Iteration = 2.3417 : 2.6907
6000th Iteration = 2.3360 : 2.6988
7000th Iteration = 2.3301 : 2.6992
8000th Iteration = 2.3248 : 2.7010
9000th Iteration = 2.3202 : 2.7091
10000th Iteration = 2.3185 : 2.7186
11000th Iteration = 2.3166 : 2.7188
12000th Iteration = 2.3156 : 2.7247
13000th Iteration = 2.3112 : 2.7260
14000th Iteration = 2.3102 : 2.7309
15000th Iteration = 2.3092 : 2.7323
16000th Iteration = 2.3086 : 2.7407
17000th Iteration = 2.3088 : 2.7374
18000th Iteration = 2.3054 : 2.7407
19000th Iteration = 2.3057 : 2.7399
20000th Iteration = 2.3032 : 2.7413


In [33]:
m.sampling(idx=torch.randint(0,d.len_char,size=(1,1)),number_of_word=1000)

Vothes herkis be ve wsesalvell
Toe.
LUtothat hean ow. iast founer e he
Anor?
MNades Cid figerto sils's o f otaufepllat at or he: wanseme'dor:
O
Bkens we aridyotsthey?


CINalinof odeldof yo en proullld llls tt adio ane ts h puto olas;
Go heal tofeplyoonmbry
An athreidind mss taryngebu, looratel tlilembn; my a n miolif sthus w'Cous orl: e pusileer y he, n thak't gel l core e?
id Houtmis h blyosete nalforisima'd fo
Tublleist.


Smatene lstf f ins
Nrecensm rerist'sulle Colenirtpre t d yomaburt ispomod h heartas sthe asony per tllyo dl m.
SI befofbably-'s theomo outstice t
nier'spe agore hice aim fa osakncend l ftencolng fed n, who sane hertreivebed ilsteashye ilnsell
O; ma wa smavis, yoe themyou hore? chan tste,-
Thoow tenand icousth ay, bk's bamave oul fofole cute
Oryou wovikershir nstre gn ouayof y o, coterl'te yo
Tit Pemmes thit ttus:
Ler twoum lfofasashait wo f ss ter p housus weld: we
Morands! noro An nfoly stet: tl the preawillolde pcerdd spothe curde ornd:
I the'lvin here, tumyenyo